In [ ]:
import sys
sys.path.append('../')

import pandas as pd
from src.agents.graph import build_fraud_graph



In [ ]:
app = build_fraud_graph()
print("graph compiled successfully")

In [ ]:
X_test = pd.read_parquet("data/processed/X_test.parquet")
Y_test = pd.read_parquet("data/processed/Y_test.parquet").iloc[:,0]



In [ ]:
import numpy as np
import joblib

In [ ]:
bundle = joblib.load('data/models/xgboost_production.pkl')
model  = bundle['model']
probs = model.predict_proba(X_test)[:,1]

In [ ]:
high_risk_idx = probs.argmax()
low_risk_idx = probs.argmin()



In [ ]:
def run_transaction(idx, label=""):
    tx_data = X_test.iloc[idx].to_dict()
    tx_id = f"TX_{idx:06d}"


    print(f"\n{'='*55}")
    print(f"Running {label} transaction: {tx_id}")
    print(f"{'='*55}")

    initial_state={
        "transaction_id": tx_id,
        "transaction_data": tx_data,
        "fraud_probability": None,
        "risk_level": None,
        "shap_explanation":None,
        "explanation_text": None,
        "decision": None,
        "policy_reasoning": None,
        "requires_human": None,
        "final_report":None,
        "processing_errors":[],
    }

    result = app.invoke(initial_state)
    report = result["final_report"]

    print(f"\n── Final Report ──")
    print(f"Transaction:  {report['transaction_id']}")
    print(f"Probability:  {report['fraud_probability']:.4f}")
    print(f"Risk level:   {report['risk_level']}")
    print(f"Decision:     {report['decision']}")
    print(f"Human review: {report['requires_human']}")
    print(f"\nPolicy reasoning:")
    print(f"  {report['policy_reasoning']}")
    print(f"\nExplanation:")
    print(f"  {report['explanation']}")

    return report





In [ ]:
high_report = run_transaction(high_risk_idx, "HIGH RISK")
low_report = run_transaction(low_risk_idx, 'LOW RISK')